# OpenTSLM signal probe — Colab runner

Runs a pretrained OpenTSLM HAR-CoT model under the probe's seven conditions and
writes the predictions to `results/`. Pick **Runtime → Change runtime type → T4 GPU** first.

Before running:
1. The base LLMs are gated on Hugging Face. Accept the licence of
   [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B)
   (or [google/gemma-3-270m](https://huggingface.co/google/gemma-3-270m) for the smaller model)
   with your Hugging Face account, and create a read token.
2. The run resumes from `results/predictions_*.jsonl`, so after a disconnect just
   re-run the cells: finished samples are skipped. Download that file before the
   session ends.

In [ ]:
!nvidia-smi -L

## 1. Install OpenTSLM at the pinned commit and clone the probe

In [ ]:
!git clone -q https://github.com/StanfordBDHG/OpenTSLM.git
!cd OpenTSLM && git checkout -q 2968f4b && pip install -q -r requirements.txt
!git clone -q https://github.com/ariannabalducci/opentslm-signal-probe.git
%cd opentslm-signal-probe

## 2. Hugging Face login (token with access to the gated base model)

In [ ]:
from huggingface_hub import login
login()

## 3. Data

Downloaded straight into OpenTSLM's own data folder, so OpenTSLM's dataset class
finds it there too and nothing is downloaded twice.

In [ ]:
DATA = "../OpenTSLM/src/data/har_cot"
!python scripts/download_data.py --data-dir {DATA}

## 4. Sanity checks

* The probe's unperturbed input must equal OpenTSLM's own (sentences and series).
* Batched generation must give the same labels as one sample at a time.

In [ ]:
!python scripts/check_against_opentslm.py --data-dir {DATA} -n 500

In [ ]:
MODEL = "OpenTSLM/llama-3.2-1b-har-sp"   # or "OpenTSLM/gemma-3-270m-pt-har-sp" (faster)
!python scripts/run_probe.py --model {MODEL} --data-dir {DATA} --batch-size 8 --check-batching

## 5. Run the probe

50 samples per class x 8 classes x 7 conditions = 2800 rationales. Start with a
smoke test (`--limit 16`), check the table, then run everything. If the batching
check above failed, use `--batch-size 1`.

In [ ]:
!python scripts/run_probe.py --model {MODEL} --data-dir {DATA} --batch-size 8 --limit 16
!python scripts/analyze.py results/predictions_{MODEL.split('/')[-1]}.jsonl

In [ ]:
!python scripts/run_probe.py --model {MODEL} --data-dir {DATA} --batch-size 8

## 6. Results

In [ ]:
!python scripts/analyze.py results/predictions_{MODEL.split('/')[-1]}.jsonl

In [ ]:
from google.colab import files
files.download(f"results/predictions_{MODEL.split('/')[-1]}.jsonl")